# Spam Classifier

Spam refers to unwanted email, often in the form of advertisements. An email that is **not** spam is called *ham*. Most email providers offer automatic spam filtering, which moves spam to a separate inbox based on the contents of the message. This requires scanning an email and deciding whether it is spam or ham: a binary classification problem.

This notebook builds a supervised **Naïve Bayes** spam classifier from scratch using only NumPy (no scikit-learn or other model-building libraries), and evaluates it on held-out data.

## Data
The training set has 1000 rows and a separate test set has 500 rows, with identical structure. Each row is an email: the first column is the label (spam `1` / ham `0`) and the other 54 columns are binary features.


In [1]:
import numpy as np

training_spam = np.loadtxt(open("data/training_spam.csv"), delimiter=",").astype(int)
print("Shape of the spam training data set:", training_spam.shape)
print(training_spam)

Shape of the spam training data set: (1000, 55)
[[1 0 0 ... 0 0 0]
 [0 0 1 ... 1 0 0]
 [0 0 0 ... 1 0 0]
 ...
 [0 0 0 ... 0 0 1]
 [1 1 1 ... 1 1 0]
 [1 0 0 ... 1 1 1]]


Your training set consists of 1000 rows and 55 columns. Each row corresponds to one email message. The first column is the _response_ variable and describes whether a message is spam `1` or ham `0`. The remaining 54 columns are _features_ used to build the classifier. These features correspond to 54 different keywords (such as "money", "free", and "receive") and special characters (such as ":", "!", and "$"). A feature has the value `1` if the keyword appears in the message and `0` otherwise.

The 500-row *test data* set contains the same 55 columns.

In [2]:
testing_spam = np.loadtxt(open("data/testing_spam.csv"), delimiter=",").astype(int)
print("Shape of the spam testing data set:", testing_spam.shape)
print(testing_spam)

Shape of the spam testing data set: (500, 55)
[[1 0 0 ... 1 1 1]
 [1 1 0 ... 1 1 1]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 1 0 0]
 [0 0 0 ... 1 0 0]]


## Naïve Bayes classifier

Naïve Bayes models the probability that a message belongs to class $c$ (spam or ham), assuming that the features (the presence of each keyword) are conditionally independent given the class:

$$p(C=c \mid \text{message}) \propto p(C=c)\prod_{i} p(w_i \mid C=c)$$

* The **class priors** $p(C=c)$ are the proportions of spam and ham in the training set.
* The **class-conditional likelihoods** $p(w_i \mid C=c)$ are estimated from the keyword counts in each class, with Laplace smoothing controlled by `alpha` so that unseen keywords never get zero probability.
* To avoid numerical underflow, everything is computed with **logarithms**, and the predicted class is the one with the highest log-posterior.

The `SpamClassifier` class below follows this scheme: `train()` computes the log priors and log likelihoods, and `predict(data)` takes an $n \times 54$ array and returns an array of $n$ predicted labels.


In [3]:
import math 

class SpamClassifier:
    def __init__(self, training_data, alpha = 1.0):
        self.training_data = training_data
        self.alpha = alpha
        
    def train(self):
        c0, c1 = math.log(np.count_nonzero(self.training_data[:, 0] == 0)/len(self.training_data)), math.log(np.count_nonzero(self.training_data[:, 0] == 1)/len(self.training_data))
        self.log_class_priors = np.array([c0,c1])

        self.log_class_conditional_likelihoods = np.array([[0]*(self.training_data.shape[1] - 1),
                                     [0]*(self.training_data.shape[1] - 1)], dtype=float)
                
        for n in range(0, self.training_data.shape[1] - 1):
    
            n_w0  = len(np.where((self.training_data[:, n+1] == 1) & (self.training_data[:, 0] == 0))[0]) + self.alpha
    
            n_w1 = len(np.where((self.training_data[:, n+1] == 1) & (self.training_data[:, 0] == 1))[0]) + self.alpha

            total_n0 = np.sum(self.training_data[self.training_data[:, 0] == 0][:,1:]) + (self.training_data.shape[1] - 1) * self.alpha
            total_n1 = np.sum(self.training_data[self.training_data[:, 0] == 1][:,1:]) + (self.training_data.shape[1] - 1) * self.alpha
    
            self.log_class_conditional_likelihoods[0, n] = math.log(n_w0/total_n0)
            self.log_class_conditional_likelihoods[1, n] = math.log(n_w1/total_n1)
        
    def predict(self, data):
        self.class_predictions = np.array(np.zeros((len(data), ), dtype=float))
        
        
        for n in range(data.shape[0]):
    
            c_0 = self.log_class_priors[0] + np.sum(self.log_class_conditional_likelihoods[0][np.where(data[n] == 1)])
            c_1 = self.log_class_priors[1] + np.sum(self.log_class_conditional_likelihoods[1][np.where(data[n] == 1)])
    
            if c_0 > c_1:
                self.class_predictions[n] = 0
            else:
                self.class_predictions[n] = 1
    
        return self.class_predictions
    

def create_classifier():
    classifier = SpamClassifier(training_data=training_spam)
    classifier.train()
    return classifier

classifier = create_classifier()

## Evaluation

The classifier is evaluated on the held-out test set. Besides accuracy, precision and recall for the spam class are reported: precision is the fraction of messages flagged as spam that really are spam, and recall is the fraction of actual spam that gets flagged.


In [4]:
testing_spam = np.loadtxt(open("data/testing_spam.csv"), delimiter=",").astype(int)
test_data = testing_spam[:, 1:]
test_labels = testing_spam[:, 0]

predictions = classifier.predict(test_data)

tp = np.sum((predictions == 1) & (test_labels == 1))
fp = np.sum((predictions == 1) & (test_labels == 0))
fn = np.sum((predictions == 0) & (test_labels == 1))

accuracy = np.mean(predictions == test_labels)
precision = tp / (tp + fp)
recall = tp / (tp + fn)

print(f"Accuracy on test data: {accuracy:.3f}")
print(f"Precision (spam): {precision:.3f}")
print(f"Recall (spam): {recall:.3f}")


Accuracy on test data: 0.898
Precision (spam): 0.843
Recall (spam): 0.915
